## Step 4 - Image features (CLIP / MetaCLIP / SigLIP / Jina-CLIP / BEiT-3)

Pick the backbone with `MODEL_KEY` — it lives in the **pip cell below**, not in the
config cell, because the `transformers` version to install depends on it. Every
backbone outputs an L2-normalised image embedding that lives in the same space as
its text encoder, so cosine search works directly. Features are namespaced per model:

```
clip-features/<MODEL_KEY>/<video_name>/<idx>.npy   # float32, ||v|| = 1
```

- `clip`   -> openai/clip-vit-large-patch14-336  (768-d, English)
- `siglip` -> google/siglip2-so400m-patch14-384  (1152-d, multilingual)  ** recommended **
- `siglip512` -> google/siglip2-so400m-patch16-512 (1152-d, hi-res: small objects/text; ~2x GPU time)
- `metaclip` -> facebook/metaclip-h14-fullcc2.5b   (1024-d, MetaCLIP **v1** — English only)
- `metaclip2` -> facebook/metaclip-2-worldwide-huge-quickgelu (1024-d, 224px, 1.86B params).
  MetaCLIP **2** worldwide: trained on 300+ languages, so its text encoder handles
  Vietnamese queries without the Gemini translation step. Needs `transformers >= 4.56`
  (the release that added the `metaclip_2` architecture) — that version is incompatible
  with `jina`, hence the conditional pin. Text side caps at 77 tokens, and its
  XLM-R tokenizer does *not* self-report that limit, so the backend passes
  `max_length=77` explicitly (see `_TEXT_MAX_LEN` in `embedding_service.py`).
  Swap in `facebook/metaclip-2-worldwide-huge-378` for 378px if small objects matter.
- `jina`   -> jinaai/jina-clip-v2                 (1024-d, multilingual incl. Vietnamese)
- `beit3`  -> beit3_large_patch16_384_retrieval   (768-d) - needs the unilm/beit3 code +
  checkpoint + `.spm`. Attach them as a Kaggle dataset and set the `BEIT3_*` paths in the
  config cell. On Kaggle, BEiT-3's modeling code needs a compatible `timm` (e.g.
  `pip install timm==0.4.12`); pin it before importing the unilm code if `create_model` fails.


In [ ]:
# MODEL_KEY đặt Ở ĐÂY (không phải cell config bên dưới) vì bản transformers phải
# khớp backbone, mà pip thì chạy trước:
#   metaclip2 -> cần >= 4.56, bản đầu tiên có kiến trúc MetaCLIP 2
#                (model_type "metaclip_2"); bản cũ hơn báo không nhận diện được.
#   jina      -> cần == 4.49: remote code jina-clip import `clip_loss`, hàm này
#                bị gỡ ở bản mới hơn -> ImportError ngay lúc load.
#   còn lại   -> >= 4.56 chạy tốt (SigLIP2 đã có từ 4.49).
# Cell config bên dưới dùng lại đúng biến này, không khai báo lần hai.
MODEL_KEY = 'metaclip2'   # clip | metaclip | metaclip2 | siglip | siglip512 | jina | beit3

_TF_PIN = 'transformers==4.49.0' if MODEL_KEY == 'jina' else 'transformers>=4.56.0'
!pip -q install -U "{_TF_PIN}" timm einops sentencepiece pillow tqdm

In [ ]:
!pip install pillow==11.3.0

In [ ]:
import os
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

In [ ]:
# MODEL_KEY đã đặt ở cell pip phía trên (bản transformers phụ thuộc nó).
KEYFRAMES_DIR = '/mnt/data2tb/code/Github/AIC/Ho-Chi-Minh-AI-Challenge/backend/media/frames/Keyframes'   # output of Step 3
OUTPUT_DIR    = f'./clip-features/{MODEL_KEY}'
BATCH_SIZE    = 64
IMAGE_EXTS    = ('.jpg', '.jpeg', '.png', '.webp')
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

# clip/siglip/jina -> HF repo id ; beit3 -> timm model name (loaded from local ckpt).
MODEL_NAME = {
    'clip':      'openai/clip-vit-large-patch14-336',
    'siglip':    'google/siglip2-so400m-patch14-384',
    'siglip512': 'google/siglip2-so400m-patch16-512',   # res cao: vật nhỏ/chữ nhỏ
    'metaclip':  'facebook/metaclip-h14-fullcc2.5b',    # MetaCLIP v1, chỉ tiếng Anh
    # MetaCLIP 2 worldwide: 1.86B tham số, ảnh 224², train 300+ ngôn ngữ (vocab
    # text 901k) nên text encoder nuốt được cả tiếng Việt chưa dịch.
    # Đổi sang '...-huge-378' nếu cần res 378 cho vật thể nhỏ (chậm hơn nhiều).
    'metaclip2': 'facebook/metaclip-2-worldwide-huge-378',
    # 'metaclip2': 'facebook/metaclip-2-worldwide-huge-quickgelu',
    'jina':      'jinaai/jina-clip-v2',
    'beit3':     'beit3_large_patch16_384_retrieval',
}[MODEL_KEY]

# --- BEiT-3 only --------------------------------------------------------------
# Needs the unilm/beit3 code + checkpoint + sentencepiece model. On Kaggle, attach
# them as a dataset and point these at the mounted paths (override as needed).
BEIT3_REPO_DIR = '/kaggle/input/beit3-assets/unilm/beit3'   # folder with modeling_finetune.py
BEIT3_CKPT     = '/kaggle/input/beit3-assets/beit3_large_itc_patch16_224.pth'
BEIT3_SPM      = '/kaggle/input/beit3-assets/beit3.spm'
# ------------------------------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('model:', MODEL_KEY, MODEL_NAME, '| device:', DEVICE)

In [ ]:
def l2(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)

N_GPU = torch.cuda.device_count()
print('GPUs:', N_GPU)

# Module bọc get_image_features để DataParallel có thể song song hóa trên nhiều GPU.
# Trả về tensor đã pool (siglip/clip đôi khi trả BaseModelOutputWithPooling thay vì tensor).
class _ImgEnc(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, **inputs):
        out = self.m.get_image_features(**inputs)
        if torch.is_tensor(out):
            return out
        if getattr(out, 'pooler_output', None) is not None:
            return out.pooler_output
        return out.last_hidden_state.mean(dim=1)

def _wrap(model):
    enc = _ImgEnc(model)
    if N_GPU > 1:
        enc = torch.nn.DataParallel(enc)          # chia batch ra các GPU theo dim 0
    return enc.to(DEVICE).eval()

# Load model + define a batched encode_image(paths) -> (N, dim) normalised
if MODEL_KEY in ('clip', 'metaclip', 'metaclip2'):
    # Cùng API get_image_features. Dùng AutoModel chứ KHÔNG phải CLIPModel:
    # metaclip2 là class MetaCLIP2Model riêng, CLIPModel.from_pretrained sẽ vỡ;
    # còn clip/metaclip thì AutoModel vẫn resolve đúng về CLIPModel.
    from transformers import AutoProcessor, AutoModel
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    enc = _wrap(model)
    @torch.no_grad()
    def encode_image(paths):
        imgs = [Image.open(p).convert('RGB') for p in paths]
        inputs = processor(images=imgs, return_tensors='pt').to(DEVICE)
        emb = enc(**inputs)
        return l2(emb.detach().cpu().numpy().astype(np.float32))

elif MODEL_KEY in ('siglip', 'siglip512'):
    # SigLIP2 repo ship GemmaTokenizer (không có vocab_file Siglip) -> AutoProcessor lỗi.
    # Chỉ cần image features nên load thẳng SiglipImageProcessor, bỏ tokenizer.
    from transformers import SiglipImageProcessor, AutoModel
    processor = SiglipImageProcessor.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    enc = _wrap(model)
    @torch.no_grad()
    def encode_image(paths):
        imgs = [Image.open(p).convert('RGB') for p in paths]
        inputs = processor(images=imgs, return_tensors='pt').to(DEVICE)
        emb = enc(**inputs)
        return l2(emb.detach().cpu().numpy().astype(np.float32))

elif MODEL_KEY == 'jina':
    # jina-clip dùng API encode_image riêng -> chạy trên 1 GPU (DataParallel không áp dụng).
    from transformers import AutoModel
    model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True).to(DEVICE).eval()
    @torch.no_grad()
    def encode_image(paths):
        imgs = [Image.open(p).convert('RGB') for p in paths]
        emb = model.encode_image(imgs, normalize_embeddings=True)
        return l2(np.asarray(emb, dtype=np.float32))

elif MODEL_KEY == 'beit3':
    # BEiT-3 dùng code + checkpoint của microsoft/unilm (không có trên HF Hub).
    # Cần attach dataset chứa unilm/beit3 + .pth + .spm rồi chỉnh các path BEIT3_* ở cell config.
    # Custom forward -> chạy 1 GPU (DataParallel không áp dụng), giống jina.
    import sys
    from torchvision import transforms

    if not os.path.isdir(BEIT3_REPO_DIR):
        raise FileNotFoundError(
            f'BEiT-3 code không thấy ở {BEIT3_REPO_DIR}. Clone https://github.com/microsoft/unilm '
            'và trỏ BEIT3_REPO_DIR tới thư mục beit3/ (xem README).'
        )
    sys.path.insert(0, BEIT3_REPO_DIR)
    import modeling_finetune  # noqa: F401 - đăng ký các timm model của BEiT-3
    import timm
    from transformers import XLMRobertaTokenizer

    model = timm.create_model(MODEL_NAME)
    ckpt = torch.load(BEIT3_CKPT, map_location='cpu')
    model.load_state_dict(ckpt.get('model', ckpt), strict=False)
    model = model.to(DEVICE).eval()

    tokenizer = XLMRobertaTokenizer(BEIT3_SPM)   # giữ lại để encode text phía backend
    # BEiT-3: resize 384², chuẩn hóa inception-style (0.5, 0.5, 0.5).
    _beit3_tf = transforms.Compose([
        transforms.Resize((384, 384), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])

    @torch.no_grad()
    def encode_image(paths):
        imgs = [Image.open(p).convert('RGB') for p in paths]
        batch = torch.stack([_beit3_tf(im) for im in imgs]).to(DEVICE)
        vision_cls, _ = model(image=batch, only_infer=True)
        return l2(vision_cls.detach().cpu().numpy().astype(np.float32))
else:
    raise ValueError(f'Unknown MODEL_KEY={MODEL_KEY!r}; '
                     'choose clip | metaclip | metaclip2 | siglip | siglip512 | jina | beit3.')

print('encoder ready')

In [ ]:
# Build the job list (flat layout, skip already-extracted .npy)
root, out_root = Path(KEYFRAMES_DIR), Path(OUTPUT_DIR)
jobs = []
for img in root.rglob('*'):
    if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
        out_path = out_root / img.relative_to(root).with_suffix('.npy')
        if not out_path.exists():
            out_path.parent.mkdir(parents=True, exist_ok=True)
            jobs.append((img, out_path))
print('to process:', len(jobs))

In [ ]:
errors = []
for i in tqdm(range(0, len(jobs), BATCH_SIZE)):
    batch = jobs[i:i + BATCH_SIZE]
    try:
        vecs = encode_image([p for p, _ in batch])
        for (_, out_path), v in zip(batch, vecs):
            np.save(out_path, v)
    except Exception:
        for img_path, out_path in batch:
            try:
                np.save(out_path, encode_image([img_path])[0])
            except Exception as e:
                errors.append((str(img_path), str(e)))
print('done. errors:', len(errors))
for p, e in errors[:10]:
    print(' ', p, '->', e)

In [ ]:
# Gộp tất cả .npy đã trích thành 1 ma trận + danh sách id (để search / FAISS)
out_root = Path(OUTPUT_DIR)
npy_files = sorted(p for p in out_root.rglob('*.npy') if p.name != 'embeddings.npy')
print('found:', len(npy_files), 'feature files')
assert npy_files, 'Chưa có file .npy nào - hãy chạy lại cell trích đặc trưng (encode) trước.'

ids   = [str(p.relative_to(out_root).with_suffix('')) for p in npy_files]  # vd: L21/L21_V017/003248
feats = np.stack([np.load(p) for p in npy_files]).astype(np.float32)
print('matrix:', feats.shape, '| dtype:', feats.dtype)

# Lưu cạnh thư mục features (đổi sang /kaggle/working/... nếu chạy trên Kaggle)
save_dir = Path(f'./clip-index/{MODEL_KEY}')
save_dir.mkdir(parents=True, exist_ok=True)
np.save(save_dir / 'embeddings.npy', feats)
(save_dir / 'ids.txt').write_text('\n'.join(ids), encoding='utf-8')
print('saved ->', save_dir / 'embeddings.npy', '&', save_dir / 'ids.txt')
